In [ ]:
Experiment 1: Latent state inference accuracy

Experiment 2: Baseline / Ablation 비교

Experiment 3: Partial observation / noise robustness

Experiment 4: Temporal adaptation

In [3]:
# ============================================================
# sacif_gpu_experiments.py
# ------------------------------------------------------------
# GPU-based experiment harness for SACIF
#
# Experiments
# 1) latent state inference accuracy
# 2) ablation / baseline comparison
# 3) partial observation & noise robustness
# 4) temporal adaptation
#
# Outputs
# - CSV files under ./sacif_results
#
# Usage
# python sacif_gpu_experiments.py --device cuda:0 --experiment all
# python sacif_gpu_experiments.py --device cpu --experiment ablation
# ============================================================

from __future__ import annotations

import os
import csv
import random
import argparse
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# Repro
# ============================================================

def set_seed(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# Constants
# ============================================================

LATENT_STATES = [
    "rested_focused",
    "sleep_deprived_slow",
    "emotionally_reactive",
    "high_noise_overloaded",
]
N_STATES = len(LATENT_STATES)

ACTIONS = [
    "low_complexity_checkin",
    "structured_guidance",
    "gentle_reflection",
    "direct_information",
    "short_action_plan",
]
N_ACTIONS = len(ACTIONS)

# observation dim:
# [latency, typing_instability, time_context, sleep_signal, emotion_signal, overload_signal]
OBS_DIM = 6

# state observation prototypes
STATE_OBS_MEANS = torch.tensor([
    [0.18, 0.18, 0.25, 0.12, 0.15, 0.20],  # rested_focused
    [0.72, 0.58, 0.62, 0.78, 0.25, 0.55],  # sleep_deprived_slow
    [0.42, 0.38, 0.45, 0.40, 0.88, 0.48],  # emotionally_reactive
    [0.62, 0.72, 0.55, 0.52, 0.42, 0.90],  # high_noise_overloaded
], dtype=torch.float32)

# action feature vectors
# [question_complexity, intervention_level, reflection_demand, information_density, response_length]
ACTION_FEATURES = torch.tensor([
    [0.2, 0.3, 0.2, 0.2, 0.3],  # low_complexity_checkin
    [0.4, 0.8, 0.3, 0.6, 0.5],  # structured_guidance
    [0.5, 0.4, 0.8, 0.3, 0.5],  # gentle_reflection
    [0.3, 0.6, 0.1, 0.8, 0.5],  # direct_information
    [0.4, 0.7, 0.2, 0.5, 0.4],  # short_action_plan
], dtype=torch.float32)

# target action profiles by latent state
STATE_TARGET_ACTION_PROFILE = torch.tensor([
    [0.6, 0.7, 0.5, 0.6, 0.5],  # rested_focused
    [0.2, 0.6, 0.2, 0.3, 0.3],  # sleep_deprived_slow
    [0.3, 0.4, 0.8, 0.2, 0.5],  # emotionally_reactive
    [0.2, 0.5, 0.2, 0.3, 0.3],  # high_noise_overloaded
], dtype=torch.float32)

# reward weights over action-profile mismatch
ENGAGEMENT_W = torch.tensor([0.30, 0.10, 0.10, 0.05, 0.45], dtype=torch.float32)
ADHERENCE_W = torch.tensor([0.10, 0.45, 0.05, 0.10, 0.30], dtype=torch.float32)
COGFIT_W = torch.tensor([0.45, 0.10, 0.25, 0.10, 0.10], dtype=torch.float32)


# ============================================================
# Config
# ============================================================

@dataclass
class Config:
    seed: int = 42
    device: str = "cuda:0"
    out_dir: str = "./sacif_results"

    train_steps: int = 600
    batch_size: int = 256
    seq_len_train: int = 8
    seq_len_eval: int = 12

    hidden_dim: int = 128
    gru_layers: int = 1
    dropout: float = 0.1
    lr: float = 3e-4
    weight_decay: float = 1e-5

    inference_loss_weight: float = 1.0
    action_loss_weight: float = 1.0

    obs_noise_train: float = 0.08
    obs_dropout_train: float = 0.05
    temporal_change_prob_train: float = 0.08

    eval_batches: int = 20
    temporal_change_prob_eval: float = 0.20

    print_every: int = 100


# ============================================================
# Device / io helpers
# ============================================================

def get_device(requested: Optional[str]) -> torch.device:
    if requested is not None:
        try:
            dev = torch.device(requested)
            if dev.type == "cuda" and not torch.cuda.is_available():
                print("[warn] CUDA requested but unavailable. Falling back to CPU.")
                return torch.device("cpu")
            return dev
        except Exception as e:
            print(f"[warn] invalid device request ({requested}): {e}. Falling back to auto.")
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def print_device_info(device: torch.device):
    print("=" * 72)
    print("torch version =", torch.__version__)
    print("cuda available=", torch.cuda.is_available())
    print("cuda count    =", torch.cuda.device_count())
    print("device        =", device)
    if device.type == "cuda":
        print("cuda name     =", torch.cuda.get_device_name(device))
        props = torch.cuda.get_device_properties(device)
        print("total memory  =", round(props.total_memory / (1024 ** 3), 2), "GB")
    else:
        print("running on CPU")
    print("=" * 72)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def save_csv(path: str, rows: List[Dict[str, object]]):
    ensure_dir(os.path.dirname(path) or ".")
    if not rows:
        return
    cols = sorted({k for r in rows for k in r.keys()})
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=cols)
        writer.writeheader()
        writer.writerows(rows)


# ============================================================
# Reward model / oracle action
# ============================================================

def compute_reward_components(
    state_idx: torch.Tensor,
    action_idx: torch.Tensor,
    device: torch.device,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    state_idx: [B,T]
    action_idx: [B,T]
    """
    target = STATE_TARGET_ACTION_PROFILE.to(device)[state_idx]  # [B,T,5]
    action = ACTION_FEATURES.to(device)[action_idx]             # [B,T,5]
    diff = torch.abs(target - action)

    engagement = 1.0 - (diff * ENGAGEMENT_W.to(device)).sum(dim=-1)
    adherence = 1.0 - (diff * ADHERENCE_W.to(device)).sum(dim=-1)
    cognitive_fit = 1.0 - (diff * COGFIT_W.to(device)).sum(dim=-1)

    engagement = engagement.clamp(0.0, 1.0)
    adherence = adherence.clamp(0.0, 1.0)
    cognitive_fit = cognitive_fit.clamp(0.0, 1.0)

    reward = 0.4 * engagement + 0.3 * adherence + 0.3 * cognitive_fit
    return engagement, adherence, cognitive_fit, reward


def compute_oracle_best_actions(device: torch.device) -> torch.Tensor:
    best = []
    for s in range(N_STATES):
        rs = []
        for a in range(N_ACTIONS):
            ss = torch.tensor([[s]], device=device)
            aa = torch.tensor([[a]], device=device)
            _, _, _, r = compute_reward_components(ss, aa, device)
            rs.append(float(r.item()))
        best.append(int(torch.tensor(rs).argmax().item()))
    return torch.tensor(best, dtype=torch.long, device=device)


# ============================================================
# Synthetic simulator
# ============================================================

class SyntheticSACIFSimulator:
    def __init__(self, device: torch.device):
        self.device = device
        self.state_obs_means = STATE_OBS_MEANS.to(device)

    def sample_batch(
        self,
        batch_size: int,
        seq_len: int,
        obs_noise: float,
        obs_dropout: float,
        temporal: bool,
        change_prob: float,
    ) -> Dict[str, torch.Tensor]:
        device = self.device

        z = torch.randint(0, N_STATES, (batch_size,), device=device)

        states: List[torch.Tensor] = []
        obs_seq: List[torch.Tensor] = []

        for t in range(seq_len):
            if t > 0 and temporal:
                change_mask = torch.rand(batch_size, device=device) < change_prob
                new_z = torch.randint(0, N_STATES, (batch_size,), device=device)
                z = torch.where(change_mask, new_z, z)

            states.append(z.clone())

            means = self.state_obs_means[z]  # [B,D]
            obs = means + obs_noise * torch.randn(batch_size, OBS_DIM, device=device)

            if obs_dropout > 0.0:
                keep_mask = (torch.rand(batch_size, OBS_DIM, device=device) > obs_dropout).float()
                obs = obs * keep_mask

            obs = obs.clamp(0.0, 1.0)
            obs_seq.append(obs)

        return {
            "obs": torch.stack(obs_seq, dim=1),      # [B,T,D]
            "states": torch.stack(states, dim=1),    # [B,T]
        }


# ============================================================
# Models
# ============================================================

class MLPBlock(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class FullSACIF(nn.Module):
    """
    belief_t = p(z_t | o_1:t)
    policy_t = pi(a_t | h_t, belief_t)
    """
    def __init__(self, obs_dim: int, hidden_dim: int, gru_layers: int, dropout: float):
        super().__init__()
        self.obs_proj = nn.Linear(obs_dim, hidden_dim)
        self.gru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=gru_layers,
            batch_first=True,
            dropout=(dropout if gru_layers >= 2 else 0.0),
        )
        self.belief_head = MLPBlock(hidden_dim, hidden_dim, N_STATES, dropout)
        self.policy_head = MLPBlock(hidden_dim + N_STATES, hidden_dim, N_ACTIONS, dropout)

    def forward(self, obs: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        x = F.relu(self.obs_proj(obs))
        h, _ = self.gru(x)
        belief_logits = self.belief_head(h)
        belief_prob = torch.softmax(belief_logits, dim=-1)
        policy_logits = self.policy_head(torch.cat([h, belief_prob], dim=-1))
        return belief_logits, policy_logits


class NoBeliefGRU(nn.Module):
    def __init__(self, obs_dim: int, hidden_dim: int, gru_layers: int, dropout: float):
        super().__init__()
        self.obs_proj = nn.Linear(obs_dim, hidden_dim)
        self.gru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=gru_layers,
            batch_first=True,
            dropout=(dropout if gru_layers >= 2 else 0.0),
        )
        self.policy_head = MLPBlock(hidden_dim, hidden_dim, N_ACTIONS, dropout)

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.obs_proj(obs))
        h, _ = self.gru(x)
        return self.policy_head(h)


class NoMemoryBelief(nn.Module):
    """
    current observation only
    belief_t = p(z_t | o_t)
    policy_t = pi(a_t | o_t, belief_t)
    """
    def __init__(self, obs_dim: int, hidden_dim: int, dropout: float):
        super().__init__()
        self.encoder = MLPBlock(obs_dim, hidden_dim, hidden_dim, dropout)
        self.belief_head = MLPBlock(hidden_dim, hidden_dim, N_STATES, dropout)
        self.policy_head = MLPBlock(hidden_dim + N_STATES, hidden_dim, N_ACTIONS, dropout)

    def forward(self, obs: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        bsz, seq, dim = obs.shape
        x = self.encoder(obs.reshape(bsz * seq, dim)).reshape(bsz, seq, -1)
        belief_logits = self.belief_head(x)
        belief_prob = torch.softmax(belief_logits, dim=-1)
        policy_logits = self.policy_head(torch.cat([x, belief_prob], dim=-1))
        return belief_logits, policy_logits


class ObsOnlyPolicy(nn.Module):
    def __init__(self, obs_dim: int, hidden_dim: int, dropout: float):
        super().__init__()
        self.policy = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, N_ACTIONS),
        )

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        bsz, seq, dim = obs.shape
        return self.policy(obs.reshape(bsz * seq, dim)).reshape(bsz, seq, N_ACTIONS)


class OracleStatePolicy(nn.Module):
    """
    upper bound: true latent state available
    """
    def __init__(self, hidden_dim: int, dropout: float):
        super().__init__()
        self.policy = nn.Sequential(
            nn.Linear(N_STATES, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, N_ACTIONS),
        )

    def forward(self, state_onehot: torch.Tensor) -> torch.Tensor:
        bsz, seq, dim = state_onehot.shape
        return self.policy(state_onehot.reshape(bsz * seq, dim)).reshape(bsz, seq, N_ACTIONS)


# ============================================================
# Train helpers
# ============================================================

def flatten_ce_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))


def train_full_sacif(
    model: FullSACIF,
    simulator: SyntheticSACIFSimulator,
    oracle_best_actions: torch.Tensor,
    cfg: Config,
    device: torch.device,
):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    model.train()

    for step in range(1, cfg.train_steps + 1):
        batch = simulator.sample_batch(
            batch_size=cfg.batch_size,
            seq_len=cfg.seq_len_train,
            obs_noise=cfg.obs_noise_train,
            obs_dropout=cfg.obs_dropout_train,
            temporal=True,
            change_prob=cfg.temporal_change_prob_train,
        )
        obs = batch["obs"]
        states = batch["states"]
        target_actions = oracle_best_actions[states]

        belief_logits, policy_logits = model(obs)
        loss_belief = flatten_ce_loss(belief_logits, states)
        loss_action = flatten_ce_loss(policy_logits, target_actions)
        loss = cfg.inference_loss_weight * loss_belief + cfg.action_loss_weight * loss_action

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        if step % cfg.print_every == 0:
            print(
                f"[FULL] step={step:04d} "
                f"loss={loss.item():.4f} "
                f"belief={loss_belief.item():.4f} "
                f"action={loss_action.item():.4f}"
            )


def train_no_memory_belief(
    model: NoMemoryBelief,
    simulator: SyntheticSACIFSimulator,
    oracle_best_actions: torch.Tensor,
    cfg: Config,
    device: torch.device,
):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    model.train()

    for step in range(1, cfg.train_steps + 1):
        batch = simulator.sample_batch(
            batch_size=cfg.batch_size,
            seq_len=cfg.seq_len_train,
            obs_noise=cfg.obs_noise_train,
            obs_dropout=cfg.obs_dropout_train,
            temporal=True,
            change_prob=cfg.temporal_change_prob_train,
        )
        obs = batch["obs"]
        states = batch["states"]
        target_actions = oracle_best_actions[states]

        belief_logits, policy_logits = model(obs)
        loss_belief = flatten_ce_loss(belief_logits, states)
        loss_action = flatten_ce_loss(policy_logits, target_actions)
        loss = cfg.inference_loss_weight * loss_belief + cfg.action_loss_weight * loss_action

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        if step % cfg.print_every == 0:
            print(
                f"[NoMemoryBelief] step={step:04d} "
                f"loss={loss.item():.4f} "
                f"belief={loss_belief.item():.4f} "
                f"action={loss_action.item():.4f}"
            )


def train_no_belief_gru(
    model: NoBeliefGRU,
    simulator: SyntheticSACIFSimulator,
    oracle_best_actions: torch.Tensor,
    cfg: Config,
    device: torch.device,
):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    model.train()

    for step in range(1, cfg.train_steps + 1):
        batch = simulator.sample_batch(
            batch_size=cfg.batch_size,
            seq_len=cfg.seq_len_train,
            obs_noise=cfg.obs_noise_train,
            obs_dropout=cfg.obs_dropout_train,
            temporal=True,
            change_prob=cfg.temporal_change_prob_train,
        )
        obs = batch["obs"]
        states = batch["states"]
        target_actions = oracle_best_actions[states]

        policy_logits = model(obs)
        loss_action = flatten_ce_loss(policy_logits, target_actions)

        opt.zero_grad()
        loss_action.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        if step % cfg.print_every == 0:
            print(f"[NoBeliefGRU] step={step:04d} action_loss={loss_action.item():.4f}")


def train_obs_only_policy(
    model: ObsOnlyPolicy,
    simulator: SyntheticSACIFSimulator,
    oracle_best_actions: torch.Tensor,
    cfg: Config,
    device: torch.device,
):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    model.train()

    for step in range(1, cfg.train_steps + 1):
        batch = simulator.sample_batch(
            batch_size=cfg.batch_size,
            seq_len=cfg.seq_len_train,
            obs_noise=cfg.obs_noise_train,
            obs_dropout=cfg.obs_dropout_train,
            temporal=True,
            change_prob=cfg.temporal_change_prob_train,
        )
        obs = batch["obs"]
        states = batch["states"]
        target_actions = oracle_best_actions[states]

        policy_logits = model(obs)
        loss_action = flatten_ce_loss(policy_logits, target_actions)

        opt.zero_grad()
        loss_action.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        if step % cfg.print_every == 0:
            print(f"[ObsOnly] step={step:04d} action_loss={loss_action.item():.4f}")


def train_oracle_state_policy(
    model: OracleStatePolicy,
    simulator: SyntheticSACIFSimulator,
    oracle_best_actions: torch.Tensor,
    cfg: Config,
    device: torch.device,
):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    model.train()

    for step in range(1, cfg.train_steps + 1):
        batch = simulator.sample_batch(
            batch_size=cfg.batch_size,
            seq_len=cfg.seq_len_train,
            obs_noise=cfg.obs_noise_train,
            obs_dropout=cfg.obs_dropout_train,
            temporal=True,
            change_prob=cfg.temporal_change_prob_train,
        )
        states = batch["states"]
        state_onehot = F.one_hot(states, num_classes=N_STATES).float()
        target_actions = oracle_best_actions[states]

        policy_logits = model(state_onehot)
        loss_action = flatten_ce_loss(policy_logits, target_actions)

        opt.zero_grad()
        loss_action.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        if step % cfg.print_every == 0:
            print(f"[OracleState] step={step:04d} action_loss={loss_action.item():.4f}")


# ============================================================
# Evaluation
# ============================================================

@torch.no_grad()
def evaluate_model(
    model_name: str,
    model,
    simulator: SyntheticSACIFSimulator,
    oracle_best_actions: torch.Tensor,
    cfg: Config,
    device: torch.device,
    obs_noise: float,
    obs_dropout: float,
    temporal: bool,
    change_prob: float,
) -> Dict[str, object]:
    if model_name != "static" and model is None:
        raise ValueError(f"model for {model_name} is None")

    if model is not None:
        model.eval()

    total_reward = 0.0
    total_eng = 0.0
    total_adh = 0.0
    total_fit = 0.0
    total_action_acc = 0.0
    total_state_acc = 0.0
    n_batches = cfg.eval_batches

    step_correct = torch.zeros(cfg.seq_len_eval, device=device)
    step_count = torch.zeros(cfg.seq_len_eval, device=device)

    for _ in range(n_batches):
        batch = simulator.sample_batch(
            batch_size=cfg.batch_size,
            seq_len=cfg.seq_len_eval,
            obs_noise=obs_noise,
            obs_dropout=obs_dropout,
            temporal=temporal,
            change_prob=change_prob,
        )

        obs = batch["obs"]
        states = batch["states"]
        target_actions = oracle_best_actions[states]

        if model_name == "full":
            belief_logits, policy_logits = model(obs)
            pred_states = belief_logits.argmax(dim=-1)
            state_acc = (pred_states == states).float()
            total_state_acc += float(state_acc.mean().item())
            step_correct += (pred_states == states).float().sum(dim=0)
            step_count += float(cfg.batch_size)

        elif model_name == "no_memory_belief":
            belief_logits, policy_logits = model(obs)
            pred_states = belief_logits.argmax(dim=-1)
            state_acc = (pred_states == states).float()
            total_state_acc += float(state_acc.mean().item())
            step_correct += (pred_states == states).float().sum(dim=0)
            step_count += float(cfg.batch_size)

        elif model_name == "no_belief_gru":
            policy_logits = model(obs)

        elif model_name == "obs_only":
            policy_logits = model(obs)

        elif model_name == "oracle_state":
            state_onehot = F.one_hot(states, num_classes=N_STATES).float()
            policy_logits = model(state_onehot)
            state_acc = torch.ones_like(states, dtype=torch.float32)
            total_state_acc += float(state_acc.mean().item())
            step_correct += state_acc.sum(dim=0)
            step_count += float(cfg.batch_size)

        elif model_name == "static":
            # fixed baseline: most frequent oracle-best action across states
            static_action = oracle_best_actions.bincount(minlength=N_ACTIONS).argmax()
            pred_actions = torch.full_like(states, fill_value=int(static_action.item()))

            eng, adh, fit, rew = compute_reward_components(states, pred_actions, device)
            action_acc = (pred_actions == target_actions).float()

            total_action_acc += float(action_acc.mean().item())
            total_reward += float(rew.mean().item())
            total_eng += float(eng.mean().item())
            total_adh += float(adh.mean().item())
            total_fit += float(fit.mean().item())
            continue

        else:
            raise ValueError(f"unknown model_name: {model_name}")

        pred_actions = policy_logits.argmax(dim=-1)

        eng, adh, fit, rew = compute_reward_components(states, pred_actions, device)
        action_acc = (pred_actions == target_actions).float()

        total_action_acc += float(action_acc.mean().item())
        total_reward += float(rew.mean().item())
        total_eng += float(eng.mean().item())
        total_adh += float(adh.mean().item())
        total_fit += float(fit.mean().item())

    out = {
        "model": model_name,
        "avg_reward": total_reward / max(1, n_batches),
        "engagement": total_eng / max(1, n_batches),
        "adherence": total_adh / max(1, n_batches),
        "cognitive_fit": total_fit / max(1, n_batches),
        "action_acc": total_action_acc / max(1, n_batches),
    }

    if model_name in ["full", "no_memory_belief", "oracle_state"]:
        out["state_acc"] = total_state_acc / max(1, n_batches)
        for t in range(cfg.seq_len_eval):
            acc_t = float((step_correct[t] / step_count[t]).item()) if step_count[t] > 0 else 0.0
            out[f"state_acc_step_{t + 1}"] = acc_t

    return out


# ============================================================
# Experiment utilities
# ============================================================

def build_models(cfg: Config, device: torch.device):
    return {
        "full": FullSACIF(OBS_DIM, cfg.hidden_dim, cfg.gru_layers, cfg.dropout).to(device),
        "no_memory_belief": NoMemoryBelief(OBS_DIM, cfg.hidden_dim, cfg.dropout).to(device),
        "no_belief_gru": NoBeliefGRU(OBS_DIM, cfg.hidden_dim, cfg.gru_layers, cfg.dropout).to(device),
        "obs_only": ObsOnlyPolicy(OBS_DIM, cfg.hidden_dim, cfg.dropout).to(device),
        "oracle_state": OracleStatePolicy(cfg.hidden_dim, cfg.dropout).to(device),
    }


# ============================================================
# Experiment 1: inference accuracy
# ============================================================

def experiment_inference(cfg: Config, device: torch.device) -> List[Dict[str, object]]:
    print("\n" + "=" * 80)
    print("Experiment 1: Latent State Inference Accuracy")
    print("=" * 80)

    simulator = SyntheticSACIFSimulator(device)
    oracle_best_actions = compute_oracle_best_actions(device)
    models = build_models(cfg, device)

    train_full_sacif(models["full"], simulator, oracle_best_actions, cfg, device)
    train_no_memory_belief(models["no_memory_belief"], simulator, oracle_best_actions, cfg, device)
    train_oracle_state_policy(models["oracle_state"], simulator, oracle_best_actions, cfg, device)

    rows = []
    for name in ["full", "no_memory_belief", "oracle_state"]:
        row = evaluate_model(
            model_name=name,
            model=models[name],
            simulator=simulator,
            oracle_best_actions=oracle_best_actions,
            cfg=cfg,
            device=device,
            obs_noise=cfg.obs_noise_train,
            obs_dropout=cfg.obs_dropout_train,
            temporal=True,
            change_prob=cfg.temporal_change_prob_eval,
        )
        row["experiment"] = "inference_accuracy"
        rows.append(row)
        print(row)

    return rows


# ============================================================
# Experiment 2: ablation / baseline
# ============================================================

def experiment_ablation(cfg: Config, device: torch.device) -> List[Dict[str, object]]:
    print("\n" + "=" * 80)
    print("Experiment 2: Baseline / Ablation")
    print("=" * 80)

    simulator = SyntheticSACIFSimulator(device)
    oracle_best_actions = compute_oracle_best_actions(device)
    models = build_models(cfg, device)

    train_full_sacif(models["full"], simulator, oracle_best_actions, cfg, device)
    train_no_memory_belief(models["no_memory_belief"], simulator, oracle_best_actions, cfg, device)
    train_no_belief_gru(models["no_belief_gru"], simulator, oracle_best_actions, cfg, device)
    train_obs_only_policy(models["obs_only"], simulator, oracle_best_actions, cfg, device)
    train_oracle_state_policy(models["oracle_state"], simulator, oracle_best_actions, cfg, device)

    rows = []
    for name in ["full", "no_memory_belief", "no_belief_gru", "obs_only", "oracle_state", "static"]:
        model = models[name] if name in models else None
        row = evaluate_model(
            model_name=name,
            model=model,
            simulator=simulator,
            oracle_best_actions=oracle_best_actions,
            cfg=cfg,
            device=device,
            obs_noise=cfg.obs_noise_train,
            obs_dropout=cfg.obs_dropout_train,
            temporal=True,
            change_prob=cfg.temporal_change_prob_eval,
        )
        row["experiment"] = "ablation"
        rows.append(row)
        print(row)

    return rows


# ============================================================
# Experiment 3: robustness
# ============================================================

def experiment_robustness(cfg: Config, device: torch.device) -> List[Dict[str, object]]:
    print("\n" + "=" * 80)
    print("Experiment 3: Partial Observation / Noise Robustness")
    print("=" * 80)

    simulator = SyntheticSACIFSimulator(device)
    oracle_best_actions = compute_oracle_best_actions(device)
    models = build_models(cfg, device)

    train_full_sacif(models["full"], simulator, oracle_best_actions, cfg, device)
    train_obs_only_policy(models["obs_only"], simulator, oracle_best_actions, cfg, device)

    conditions = [
        ("low", 0.03, 0.00),
        ("medium", 0.08, 0.10),
        ("high", 0.15, 0.30),
    ]

    rows = []
    for cond_name, noise, dropout in conditions:
        for name in ["full", "obs_only"]:
            row = evaluate_model(
                model_name=name,
                model=models[name],
                simulator=simulator,
                oracle_best_actions=oracle_best_actions,
                cfg=cfg,
                device=device,
                obs_noise=noise,
                obs_dropout=dropout,
                temporal=True,
                change_prob=cfg.temporal_change_prob_eval,
            )
            row["experiment"] = "robustness"
            row["condition"] = cond_name
            row["obs_noise"] = noise
            row["obs_dropout"] = dropout
            rows.append(row)
            print(row)

    return rows


# ============================================================
# Experiment 4: temporal adaptation
# ============================================================

def experiment_temporal(cfg: Config, device: torch.device) -> List[Dict[str, object]]:
    print("\n" + "=" * 80)
    print("Experiment 4: Temporal Adaptation")
    print("=" * 80)

    simulator = SyntheticSACIFSimulator(device)
    oracle_best_actions = compute_oracle_best_actions(device)
    models = build_models(cfg, device)

    train_full_sacif(models["full"], simulator, oracle_best_actions, cfg, device)
    train_no_memory_belief(models["no_memory_belief"], simulator, oracle_best_actions, cfg, device)

    rows = []
    for name in ["full", "no_memory_belief"]:
        row = evaluate_model(
            model_name=name,
            model=models[name],
            simulator=simulator,
            oracle_best_actions=oracle_best_actions,
            cfg=cfg,
            device=device,
            obs_noise=cfg.obs_noise_train,
            obs_dropout=cfg.obs_dropout_train,
            temporal=True,
            change_prob=0.35,
        )
        row["experiment"] = "temporal_adaptation"
        row["change_prob"] = 0.35
        rows.append(row)
        print(row)

    return rows


# ============================================================
# Args / main
# ============================================================

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--device", type=str, default="cuda:0")
    p.add_argument(
        "--experiment",
        type=str,
        default="all",
        choices=["inference", "ablation", "robustness", "temporal", "all"],
    )
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--out-dir", type=str, default="./sacif_results")
    p.add_argument("--train-steps", type=int, default=600)
    p.add_argument("--batch-size", type=int, default=256)
    p.add_argument("--seq-len-train", type=int, default=8)
    p.add_argument("--seq-len-eval", type=int, default=12)
    p.add_argument("--hidden-dim", type=int, default=128)
    p.add_argument("--lr", type=float, default=3e-4)
    p.add_argument("--print-every", type=int, default=100)
    args, _ = p.parse_known_args()
    return args


def main():
    args = parse_args()

    cfg = Config(
        seed=args.seed,
        device=args.device,
        out_dir=args.out_dir,
        train_steps=args.train_steps,
        batch_size=args.batch_size,
        seq_len_train=args.seq_len_train,
        seq_len_eval=args.seq_len_eval,
        hidden_dim=args.hidden_dim,
        lr=args.lr,
        print_every=args.print_every,
    )

    set_seed(cfg.seed)
    device = get_device(cfg.device)
    print_device_info(device)
    ensure_dir(cfg.out_dir)

    all_rows: List[Dict[str, object]] = []

    if args.experiment in ["inference", "all"]:
        rows = experiment_inference(cfg, device)
        save_csv(os.path.join(cfg.out_dir, "experiment_inference.csv"), rows)
        all_rows.extend(rows)

    if args.experiment in ["ablation", "all"]:
        rows = experiment_ablation(cfg, device)
        save_csv(os.path.join(cfg.out_dir, "experiment_ablation.csv"), rows)
        all_rows.extend(rows)

    if args.experiment in ["robustness", "all"]:
        rows = experiment_robustness(cfg, device)
        save_csv(os.path.join(cfg.out_dir, "experiment_robustness.csv"), rows)
        all_rows.extend(rows)

    if args.experiment in ["temporal", "all"]:
        rows = experiment_temporal(cfg, device)
        save_csv(os.path.join(cfg.out_dir, "experiment_temporal.csv"), rows)
        all_rows.extend(rows)

    if all_rows:
        save_csv(os.path.join(cfg.out_dir, "experiment_all.csv"), all_rows)

    print("\nDone.")
    print("Results saved to:", cfg.out_dir)


if __name__ == "__main__":
    main()

torch version = 2.10.0+cu128
cuda available= True
cuda count    = 1
device        = cuda:0
cuda name     = NVIDIA L40S
total memory  = 44.52 GB

Experiment 1: Latent State Inference Accuracy
[FULL] step=0100 loss=0.1558 belief=0.1033 action=0.0525
[FULL] step=0200 loss=0.0365 belief=0.0189 action=0.0176
[FULL] step=0300 loss=0.0130 belief=0.0084 action=0.0047
[FULL] step=0400 loss=0.0072 belief=0.0046 action=0.0026
[FULL] step=0500 loss=0.0061 belief=0.0046 action=0.0015
[FULL] step=0600 loss=0.0097 belief=0.0066 action=0.0031
[NoMemoryBelief] step=0100 loss=0.1728 belief=0.1119 action=0.0609
[NoMemoryBelief] step=0200 loss=0.0632 belief=0.0470 action=0.0162
[NoMemoryBelief] step=0300 loss=0.0337 belief=0.0268 action=0.0069
[NoMemoryBelief] step=0400 loss=0.0188 belief=0.0153 action=0.0035
[NoMemoryBelief] step=0500 loss=0.0140 belief=0.0120 action=0.0020
[NoMemoryBelief] step=0600 loss=0.0127 belief=0.0112 action=0.0015
[OracleState] step=0100 action_loss=0.3219
[OracleState] step=020

In [1]:
!nvidia-smi

Sun Mar 15 21:50:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.127.08             Driver Version: 550.127.08     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40S                    Off |   00000000:BE:00.0 Off |                    0 |
| N/A   63C    P0            112W /  350W |    2301MiB /  46068MiB |     20%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----